In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv("/content/drive/MyDrive/ml /mbti_1.csv")

df.head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...


In [4]:
#CLEAN FUNCTION
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\|\|\|", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_posts'] = df['posts'].apply(clean_text)

In [5]:
#TF-IDF FEATURES

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X = tfidf.fit_transform(df['clean_posts'])


In [6]:
#ENCODE LABEL
le = LabelEncoder()
y = le.fit_transform(df['type'])


In [7]:
#TRAIN TEST SPLIT (80:20)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

In [8]:
#GRID SEARCH

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)

grid.fit(X_train, y_train)

rf = grid.best_estimator_
rf.fit(X_train, y_train)


RandomForestClassifier(max_depth=30, min_samples_split=5, n_estimators=500,
                       random_state=42)

In [20]:
#EVALUATE

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.5693430656934306
Weighted F1: 0.5182143294244576
              precision    recall  f1-score   support

        ENFJ       1.00      0.02      0.03        57
        ENFP       0.77      0.33      0.46       203
        ENTJ       1.00      0.03      0.06        69
        ENTP       0.71      0.34      0.46       206
        ESFJ       0.00      0.00      0.00        13
        ESFP       0.00      0.00      0.00        14
        ESTJ       0.00      0.00      0.00        12
        ESTP       0.00      0.00      0.00        27
        INFJ       0.58      0.71      0.64       441
        INFP       0.47      0.91      0.62       550
        INTJ       0.68      0.58      0.63       327
        INTP       0.62      0.77      0.69       391
        ISFJ       1.00      0.08      0.15        50
        ISFP       1.00      0.05      0.09        81
        ISTJ       0.67      0.03      0.06        61
        ISTP       0.81      0.26      0.39       101

    accuracy       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
#CREATE INPUT
import scipy.sparse as sp

feature_names = tfidf.get_feature_names_out()

user_df = pd.DataFrame(
    sp.csr_matrix(tfidf.transform([" "])).toarray(),
    columns=feature_names
)

user_X = sp.csr_matrix(user_df.values)

In [19]:
#PREDICT

prediction = rf.predict(user_X)
predicted_type = le.inverse_transform(prediction)

print("\n✅ Predicted MBTI Type:", predicted_type[0])



✅ Predicted MBTI Type: INFP


In [23]:
#SMART RECOMMENDATION

filtered = df[df['type'] == predicted_type[0]].copy()
filtered['clean_posts'] = filtered['clean_posts'].astype(str)

def calculate_score(row):
    score = 0
    for word in user_clean.split():
        if word in row['clean_posts']:
            score += 1
    return score

filtered['score'] = filtered.apply(calculate_score, axis=1)
filtered = filtered.sort_values(by='score', ascending=False)


In [24]:
#OP 3 RESULTS
print("\n Top Similar Profiles:\n")

top_results = filtered.head(3)

for i, row in top_results.iterrows():
    print("🔹 MBTI Type:", row['type'])
    print("🔹 Post Preview:", row['posts'][:100], "...")
    print("🔹 Score:", row['score'])
    print("-" * 40)



 Top Similar Profiles:

🔹 MBTI Type: INFP
🔹 Post Preview: 'That's an opinion, not a fact.  That being said, I believe in the inherent right of everyone to be  ...
🔹 Score: 12
----------------------------------------
🔹 MBTI Type: INFP
🔹 Post Preview: 'I am an INFP and that sounds way too stereotypical. And you can become who you want to be without t ...
🔹 Score: 12
----------------------------------------
🔹 MBTI Type: INFP
🔹 Post Preview: 'Hello! :) I feel like I can relate to you on more levels than one.  My high school years were devot ...
🔹 Score: 12
----------------------------------------
